# Phân tích SQL – Dự đoán phí vận chuyển đơn hàng thương mại điện tử Olist
**Nhóm 8 · Lớp AI2107 · Môn ADY201M · FALL26**

---
## 1. Mục tiêu của bước SQL

Sau bước làm sạch (Phần 3), nhóm dùng SQL để **khám phá bảng dữ liệu sạch trước khi xây mô hình**. Bước này hỗ trợ các câu hỏi kinh doanh ở Phần 1:

- **Câu hỏi 1.** Trong các yếu tố như cân nặng, thể tích, khoảng cách giữa người bán và người mua, giá sản phẩm, yếu tố nào có mối liên hệ mạnh nhất với phí vận chuyển (`freight_value`)?
- **Câu hỏi 2.** Có thể dự đoán phí vận chuyển cho từng dòng sản phẩm với sai số trung bình bao nhiêu BRL, và các đặc điểm trên giải thích được bao nhiêu phần biến động của phí?
- **Câu hỏi 3.** Quan hệ giữa các đặc điểm trên và phí vận chuyển có mang tính tuyến tính hay không, tức là mô hình phi tuyến (Random Forest) có dự đoán tốt hơn mô hình tuyến tính (Linear Regression) hay không?

Bước SQL chỉ làm một việc: **tính phí vận chuyển trung bình theo nhóm** của từng yếu tố nêu trong Câu hỏi 1 (cân nặng, thể tích, khoảng cách, giá), để mô tả dữ liệu. SQL **không xếp hạng mức độ quan trọng** của các yếu tố vì các yếu tố liên quan với nhau (ví dụ cân nặng và thể tích); việc đó thuộc EDA (Phần 5) và mô hình (Phần 6). Câu hỏi 2 và Câu hỏi 3 được trả lời ở bước hồi quy (Phần 6) bằng MAE, RMSE, R² và so sánh hai mô hình.

## 2. Dữ liệu sử dụng

- **Nguồn:** Brazilian E-Commerce Public Dataset by Olist (Kaggle), giấy phép CC BY-NC-SA 4.0.
- **Bảng dùng trong SQL:** `olist_clean` (108.937 dòng, 12 cột) do Phần 3 xuất ra (tệp `olist_clean.csv`): chỉ giữ đơn đã giao, giai đoạn 01/2017–08/2018, bỏ dòng thiếu tọa độ, cân nặng hoặc kích thước, bỏ dòng có cân nặng bằng 0 và phí vận chuyển bằng 0.
- **Đơn vị quan sát:** một sản phẩm trong một đơn hàng, xác định bởi cặp (`order_id`, `order_item_id`).

| Cột | Ý nghĩa |
|---|---|
| `order_id`, `order_item_id` | Khóa của dòng (đơn hàng, số thứ tự sản phẩm trong đơn) |
| `seller_id` | Mã người bán |
| `order_purchase_timestamp` | Thời điểm đặt hàng (tháng đặt hàng được lấy từ cột này bằng `strftime` ở Query 7) |
| `price` | Giá bán của sản phẩm (BRL) |
| `weight_g` | Cân nặng sản phẩm (gam) |
| `volume_cm3` | Thể tích sản phẩm (dài × cao × rộng) |
| `distance_km` | Khoảng cách đường chim bay giữa người bán và khách (km), tính theo công thức Haversine từ tọa độ mã bưu chính |
| `category` | Danh mục sản phẩm (tiếng Anh; `unknown` nếu thiếu) |
| `order_n_items` | Số sản phẩm của đơn gốc (đếm trước khi loại dòng) |
| `same_state` | 1 nếu người bán và khách cùng bang, 0 nếu khác bang |
| `freight_value` | Phí vận chuyển (BRL), **biến cần dự đoán** |

## 3. Chuẩn bị môi trường
Nạp file dữ liệu sạch vào SQLite (cơ sở dữ liệu chạy trong bộ nhớ) và khai báo hàm `sql()` để chạy câu lệnh.

In [1]:
import sqlite3, pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Tệp do Phần 3 xuất ra. Colab: kéo olist_clean.csv vào /content/ rồi giữ nguyên dòng dưới.
# Trong repo: '../data/cleaned/olist_clean.csv'
CSV_PATH = '/content/olist_clean.csv'
df = pd.read_csv(CSV_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('olist_clean', conn, index=False, if_exists='replace')

def sql(query):
    """Chạy một câu SQL và trả về bảng kết quả"""
    return pd.read_sql_query(query, conn)

print(df.shape)

(108937, 12)


### Kiểm tra dữ liệu đã nạp
Xác nhận bảng nạp vào đúng với kết quả của bước làm sạch.

In [2]:
print('Số dòng, số cột        :', df.shape)
print('Tổng ô thiếu           :', df.isna().sum().sum())
print('Số dòng trùng khóa      :', df.duplicated(['order_id', 'order_item_id']).sum())
print('Giai đoạn đặt hàng      :', df['order_purchase_timestamp'].str[:7].min(), 'đến', df['order_purchase_timestamp'].str[:7].max())
print('Dòng category = unknown :', (df['category'] == 'unknown').sum())
assert df.shape == (108937, 12), 'Nạp nhầm tệp: cần bản 12 cột, 108.937 dòng'


Số dòng, số cột        : (108937, 12)
Tổng ô thiếu           : 0
Số dòng trùng khóa      : 0
Giai đoạn đặt hàng      : 2017-01 đến 2018-08
Dòng category = unknown : 1532


## 4. Các truy vấn

Bảy truy vấn: Query 1 mô tả chung; Query 2 đến 5 lần lượt xem phí theo cân nặng, thể tích, khoảng cách và giá; Query 6 so sánh cùng bang và khác bang; Query 7 xem cân nặng và khoảng cách cùng lúc. Các yếu tố được chia thành nhóm bằng mốc tròn, dễ đọc, rồi tính số dòng và phí trung bình. Cột `so_dong` là số dòng (sản phẩm trong đơn), không phải số đơn. Mốc chia không được chọn để tối ưu kết quả.

---
## Query 1 — Tổng quan dữ liệu và phí vận chuyển

**Câu hỏi phân tích**: Dữ liệu có bao nhiêu dòng, bao nhiêu đơn hàng? Phí vận chuyển nhỏ nhất, lớn nhất và trung bình là bao nhiêu?
**Mục đích**: Nắm quy mô dữ liệu và đặc điểm của biến mục tiêu `freight_value` trước khi đi vào từng yếu tố ảnh hưởng.

In [3]:
print("=== 1. TỔNG QUAN DỮ LIỆU VÀ PHÍ VẬN CHUYỂN ===")
print(sql("""
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT order_id) AS so_don_duy_nhat,
       MIN(freight_value) AS phi_nho_nhat,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB,
       MAX(freight_value) AS phi_lon_nhat
FROM olist_clean;
"""))


=== 1. TỔNG QUAN DỮ LIỆU VÀ PHÍ VẬN CHUYỂN ===
   so_dong  so_don_duy_nhat  phi_nho_nhat  phi_ship_TB  phi_lon_nhat
0   108937            95377          0.01        20.01        409.68


**Nhận xét**: Dữ liệu có **108.937** dòng thuộc **95.377** đơn hàng. Phí trung bình là 20,01 BRL nhưng phí cao nhất lên tới 409,68 BRL (gấp khoảng **20 lần** mức trung bình), tức phí lệch phải mạnh. Vì vậy khi đánh giá mô hình cần báo MAE cùng sai số theo từng nhóm, không chỉ dựa vào mức trung bình.

---
## Query 2 — Phí ship theo nhóm cân nặng

**Câu hỏi phân tích**: Hàng càng nặng thì phí vận chuyển có tăng không?
**Mục đích**: Mô tả phí theo nhóm cân nặng (`weight_g`).

In [4]:
print("=== 2. PHÍ SHIP THEO NHÓM CÂN NẶNG ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN weight_g < 500 THEN '1. Rất nhẹ (<500g)'
               WHEN weight_g < 2000 THEN '2. Nhẹ (500-2000g)'
               WHEN weight_g < 5000 THEN '3. Trung bình (2-5kg)'
               ELSE '4. Nặng (>5kg)' END AS nhom_can_nang
    FROM olist_clean
)
SELECT nhom_can_nang,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_can_nang
ORDER BY nhom_can_nang;
"""))


=== 2. PHÍ SHIP THEO NHÓM CÂN NẶNG ===
           nhom_can_nang  so_dong  phi_ship_TB
0     1. Rất nhẹ (<500g)    43532        15.17
1     2. Nhẹ (500-2000g)    40632        17.89
2  3. Trung bình (2-5kg)    11220        22.81
3         4. Nặng (>5kg)    13553        39.62


**Nhận xét**: Phí trung bình tăng từ 15,17 BRL (dưới 500g) lên 39,62 BRL (trên 5kg), gấp **2,6 lần**, và tăng dần qua từng nhóm.

---
## Query 3 — Phí ship theo nhóm thể tích

**Câu hỏi phân tích**: Hàng càng cồng kềnh thì phí vận chuyển có tăng không?
**Mục đích**: Mô tả phí theo nhóm thể tích (`volume_cm3`).

In [5]:
print("=== 3. PHÍ SHIP THEO NHÓM THỂ TÍCH ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN volume_cm3 < 2000 THEN '1. Nhỏ (<2.000 cm3)'
               WHEN volume_cm3 < 5000 THEN '2. Vừa (2.000-5.000)'
               WHEN volume_cm3 < 15000 THEN '3. Lớn (5.000-15.000)'
               ELSE '4. Rất lớn (>15.000)' END AS nhom_the_tich
    FROM olist_clean
)
SELECT nhom_the_tich,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_the_tich
ORDER BY nhom_the_tich;
"""))


=== 3. PHÍ SHIP THEO NHÓM THỂ TÍCH ===


           nhom_the_tich  so_dong  phi_ship_TB
0    1. Nhỏ (<2.000 cm3)    16927        14.91
1   2. Vừa (2.000-5.000)    30224        16.27
2  3. Lớn (5.000-15.000)    29070        18.61
3   4. Rất lớn (>15.000)    32716        27.36


**Nhận xét**: Phí trung bình tăng từ 14,91 BRL (dưới 2.000 cm3) lên 27,36 BRL (trên 15.000 cm3), gấp **1,8 lần**; mức tăng rõ nhất ở nhóm rất lớn.

---
## Query 4 — Phí ship theo nhóm khoảng cách

**Câu hỏi phân tích**: Khoảng cách giao hàng (người bán đến người mua) ảnh hưởng thế nào đến phí vận chuyển?
**Mục đích**: Mô tả phí theo nhóm khoảng cách (`distance_km`).

In [6]:
print("=== 4. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN distance_km < 200 THEN '1. Gần (<200km)'
               WHEN distance_km < 800 THEN '2. Trung bình (200-800km)'
               WHEN distance_km < 1500 THEN '3. Xa (800-1500km)'
               ELSE '4. Rất xa (>1500km)' END AS nhom_khoang_cach
    FROM olist_clean
)
SELECT nhom_khoang_cach,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_khoang_cach
ORDER BY nhom_khoang_cach;
"""))


=== 4. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===
            nhom_khoang_cach  so_dong  phi_ship_TB
0            1. Gần (<200km)    28096        12.66
1  2. Trung bình (200-800km)    54032        19.80
2         3. Xa (800-1500km)    17099        23.74
3        4. Rất xa (>1500km)     9710        35.91


**Nhận xét**: Phí trung bình tăng từ 12,66 BRL (dưới 200km) lên 35,91 BRL (trên 1500km), gấp **2,8 lần**, và tăng đều qua từng nhóm.

---
## Query 5 — Phí ship theo nhóm giá bán

**Câu hỏi phân tích**: Sản phẩm giá cao hơn thì phí vận chuyển có cao hơn không?
**Mục đích**: Mô tả phí theo nhóm giá (`price`).

In [7]:
print("=== 5. PHÍ SHIP THEO NHÓM GIÁ BÁN ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN price < 50 THEN '1. Rẻ (<50)'
               WHEN price < 100 THEN '2. 50-100'
               WHEN price < 200 THEN '3. 100-200'
               ELSE '4. Đắt (>200)' END AS nhom_gia
    FROM olist_clean
)
SELECT nhom_gia,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_gia
ORDER BY nhom_gia;
"""))


=== 5. PHÍ SHIP THEO NHÓM GIÁ BÁN ===


        nhom_gia  so_dong  phi_ship_TB
0    1. Rẻ (<50)    37965        14.73
1      2. 50-100    31941        18.01
2     3. 100-200    26093        23.13
3  4. Đắt (>200)    12938        34.16


**Nhận xét**: Phí trung bình tăng từ 14,73 BRL (giá dưới 50) lên 34,16 BRL (giá trên 200), gấp **2,3 lần**. Giá cao thường đi với hàng lớn và nặng, nên đây chưa phải tác động riêng của giá.

---
## Query 6 — Cùng bang và khác bang

**Câu hỏi phân tích**: Giao hàng trong cùng một bang có rẻ hơn khác bang không?
**Mục đích**: Mô tả phí theo biến `same_state`, kèm khoảng cách trung bình của hai nhóm.

In [8]:
print("=== 6. CÙNG BANG VS KHÁC BANG ===")
print(sql("""
SELECT CASE WHEN same_state = 1 THEN 'Cùng bang' ELSE 'Khác bang' END AS nhom,
       COUNT(*) AS so_dong,
       ROUND(AVG(distance_km), 0) AS khoang_cach_TB,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM olist_clean
GROUP BY same_state;
"""))


=== 6. CÙNG BANG VS KHÁC BANG ===


        nhom  so_dong  khoang_cach_TB  phi_ship_TB
0  Khác bang    69330           848.0        23.74
1  Cùng bang    39607           154.0        13.48


**Nhận xét**: Đơn khác bang có phí trung bình cao hơn **76%** (23,74 so với 13,48 BRL), nhưng khoảng cách trung bình cũng xa hơn nhiều (848km so với 154km). `same_state` liên quan chặt với khoảng cách nên khó nói phần chênh phí đến từ đâu.

---
## Query 7 — Cân nặng và khoảng cách cùng lúc

**Câu hỏi phân tích**: Khoảng cách làm phí tăng như nhau ở hàng nhẹ và hàng nặng không?
**Mục đích**: Xem hai yếu tố tác động riêng lẻ hay cùng nhau, làm bằng chứng mô tả cho Câu hỏi 3 (quan hệ có tuyến tính không).

In [9]:
print("=== 7. PHÍ SHIP THEO CÂN NẶNG x KHOẢNG CÁCH ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN weight_g < 500 THEN '1. <500g'
                WHEN weight_g < 2000 THEN '2. 500-2000g'
                WHEN weight_g < 5000 THEN '3. 2-5kg'
                ELSE '4. >5kg' END AS nhom_can_nang,
           CASE WHEN distance_km < 200 THEN '1. <200km'
                WHEN distance_km < 800 THEN '2. 200-800km'
                WHEN distance_km < 1500 THEN '3. 800-1500km'
                ELSE '4. >1500km' END AS nhom_khoang_cach
    FROM olist_clean
)
SELECT nhom_can_nang, nhom_khoang_cach,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_can_nang, nhom_khoang_cach
ORDER BY nhom_can_nang, nhom_khoang_cach;
"""))


=== 7. PHÍ SHIP THEO CÂN NẶNG x KHOẢNG CÁCH ===


   nhom_can_nang nhom_khoang_cach  so_dong  phi_ship_TB
0       1. <500g        1. <200km    12673         9.17
1       1. <500g     2. 200-800km    20069        15.39
2       1. <500g    3. 800-1500km     6898        18.32
3       1. <500g       4. >1500km     3892        27.95
4   2. 500-2000g        1. <200km     9253        11.75
5   2. 500-2000g     2. 200-800km    21418        17.15
6   2. 500-2000g    3. 800-1500km     6305        20.93
7   2. 500-2000g       4. >1500km     3656        32.49
8       3. 2-5kg        1. <200km     2706        15.14
9       3. 2-5kg     2. 200-800km     5615        22.04
10      3. 2-5kg    3. 800-1500km     1877        26.86
11      3. 2-5kg       4. >1500km     1022        39.94
12       4. >5kg        1. <200km     3464        25.93
13       4. >5kg     2. 200-800km     6930        38.92
14       4. >5kg    3. 800-1500km     2019        48.10
15       4. >5kg       4. >1500km     1140        70.47


**Nhận xét**: Ở mọi nhóm khoảng cách, hàng nặng đều đắt hơn hàng nhẹ. Khi đi từ gần (dưới 200km) đến rất xa (trên 1500km), hàng dưới 500g tăng từ 9,17 lên 27,95 BRL (**+18,8 BRL**, gấp 3,0 lần), còn hàng trên 5kg tăng từ 25,93 lên 70,47 BRL (**+44,5 BRL**, gấp 2,7 lần). Mức tăng tuyệt đối lớn hơn ở hàng nặng, nên hai yếu tố không cộng đơn giản mà có tương tác. Ô nhỏ nhất có 1.022 dòng. Đây chỉ là bằng chứng mô tả cho Câu hỏi 3; việc so sánh mô hình tuyến tính và phi tuyến thuộc Phần 6.

---
## 5. Tổng hợp phát hiện

| Query | Phát hiện | Dùng cho bước sau |
|---|---|---|
| 1 | Phí trung bình 20,01 BRL, cao nhất 409,68 BRL (gấp khoảng 20 lần) | Mốc so sánh cho sai số mô hình ở Phần 6 |
| 2 | Phí tăng từ 15,17 lên 39,62 BRL theo cân nặng | Đưa `weight_g` vào mô hình |
| 3 | Phí tăng từ 14,91 lên 27,36 BRL theo thể tích | Đưa `volume_cm3` vào mô hình; liên quan với cân nặng |
| 4 | Phí tăng từ 12,66 lên 35,91 BRL theo khoảng cách | Đưa `distance_km` vào mô hình |
| 5 | Phí tăng từ 14,73 lên 34,16 BRL theo giá; giá đi cùng hàng lớn, nặng | Đưa `price` vào mô hình, chưa coi là tác động riêng |
| 6 | Cùng bang 13,48 BRL, khác bang 23,74 BRL (khoảng cách 154km so với 848km) | `same_state` liên quan chặt với khoảng cách; không đưa vào mô hình ở Phần 6 |
| 7 | Khoảng cách làm phí tăng +18,8 BRL (hàng dưới 500g) và +44,5 BRL (hàng trên 5kg) | Có tương tác cân nặng và khoảng cách; so sánh mô hình tuyến tính và phi tuyến (Câu hỏi 3) |

Cả bốn yếu tố đều đi cùng phí cao hơn. SQL chưa cho biết yếu tố nào quan trọng hơn: việc đó do tương quan (Phần 5) và độ quan trọng của đặc trưng (Phần 6) trả lời.

## 6. Hạn chế và bước tiếp theo

**Hạn chế:**
- Chỉ so sánh trung bình theo nhóm, không tách được tác động riêng của từng yếu tố khi các yếu tố liên quan với nhau; kết quả chỉ mô tả, chưa phải kết luận về nguyên nhân hay mức độ quan trọng.
- Phí trung bình bị kéo lên bởi một số phí rất lớn (cao nhất 409,68 BRL), nên trung vị cần xem thêm ở EDA.
- Chưa xác định được phí của đơn nhiều sản phẩm là phí riêng của sản phẩm hay phí của cả đơn được chia (Phần 3, mục 9.2).
- Khoảng cách tính từ tọa độ mã bưu chính theo đường chim bay, không phải quãng đường thực.

**Bước tiếp theo:** EDA bằng Python (Phần 5: biểu đồ, tương quan Pearson) rồi hồi quy (Phần 6: Linear Regression và Random Forest, đo bằng MAE, RMSE, R²).

In [10]:
conn.close()
print('\nĐã đóng kết nối SQLite.')



Đã đóng kết nối SQLite.
